# 04 — HybridRAG: GraphRAG vs VectorRAG vs Hybrid

**Reasoning-LM variant.** Evaluates three retrieval regimes against a shared
closed-book baseline on the Alzheimer's slice of MIRAGE.

| Panel | Strategy key | Evidence supplied to the generator |
|---|---|---|
| A | `closed_book` | none — parametric knowledge only |
| B | `graphrag`    | KG triples: bridge edges + per-entity neighbourhoods |
| C | `vectorrag`   | top-*k* FAISS chunks from the AD PubMed abstracts |
| D | `hybridrag`   | both, reciprocal-rank fused into one context block |

Both retrievers index **the same PubMed corpus** — the graph was extracted from
those abstracts (`01_...ipynb`) and the vector index was built over them
(`03_...ipynb`). Corpus is therefore held constant and only the *retrieval
method* varies, which is what makes the B/C/D comparison interpretable.

---

## Running this as a Hugging Face Job

This notebook is written to run as a **`hf jobs uv run`** script as well as
interactively. Convert and launch:

```bash
jupyter nbconvert --to script 04_hybridrag_and_evaluation.ipynb
hf jobs uv run --flavor l4x1 --timeout 6h \
    --secrets HF_TOKEN \
    --env KG_REPO=NathanPereira/alzheimers-kg \
    --env GENERATOR=google/medgemma-27b-text-it \
    --env LIMIT=0 \
    04_hybridrag_and_evaluation.py
```

**Flavour guidance.** `l4x1` (24 GB, Ada) holds the 4-bit 27B on one card and
supports bf16, so it avoids the fp16-overflow and cross-device sharding failures
that appear on 2×T4. `a10g-large` is the fallback. Every knob below reads from
an environment variable with a sensible default, so no code edits are needed
between runs.

## 0 · Dependencies

In [ ]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "torch",
#     "transformers",
#     "accelerate",
#     "bitsandbytes",
#     "sentence-transformers",
#     "faiss-cpu",
#     "networkx",
#     "numpy",
#     "matplotlib",
#     "huggingface_hub",
# ]
# ///
# The block above is the PEP-723 header hf jobs uv run reads. When running
# interactively in a notebook instead, install the same set:
import importlib, subprocess, sys
if importlib.util.find_spec("faiss") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers", "accelerate", "bitsandbytes",
                    "sentence-transformers", "faiss-cpu", "networkx",
                    "huggingface_hub"], check=False)
print("deps ready")

## 1 · Config

Everything is environment-driven so the same file runs unchanged as a Job.
`STRATEGIES` defines the comparison; `closed_book` must stay first because it is
the McNemar reference and the flip-analysis baseline. Measuring the baseline
inside the same run — same model, same precision, same questions — is what makes
the deltas attributable to retrieval rather than to a change of generator.

In [ ]:
import os, re, json, time, math, pickle, csv, urllib.request
from collections import Counter, defaultdict
from difflib import get_close_matches
import numpy as np, torch

# ---- repos / model --------------------------------------------------------
KG_REPO      = os.environ.get("KG_REPO",      "NathanPereira/alzheimers-kg")
RESULTS_REPO = os.environ.get("RESULTS_REPO", KG_REPO)
GENERATOR    = os.environ.get("GENERATOR",    "google/medgemma-27b-text-it")
ENCODER      = os.environ.get("ENCODER",      "NeuML/pubmedbert-base-embeddings")

# ---- what to run ----------------------------------------------------------
STRATEGIES = tuple(s.strip() for s in os.environ.get(
    "STRATEGIES", "closed_book,graphrag,vectorrag,hybridrag").split(",") if s.strip())
LIMIT      = int(os.environ.get("LIMIT", "0"))      # 0 = all questions
SC_SAMPLES = int(os.environ.get("SC_SAMPLES", "1")) # >1 enables self-consistency
SC_TEMP    = float(os.environ.get("SC_TEMP", "0.7"))

# ---- retrieval ------------------------------------------------------------
MAX_ENTITIES  = int(os.environ.get("MAX_ENTITIES", "3"))
FACTS_PER_ENT = int(os.environ.get("FACTS_PER_ENT", "8"))
MIN_CONF      = int(os.environ.get("MIN_CONF", "6"))
VEC_TOPK      = int(os.environ.get("VEC_TOPK", "5"))
RRF_K         = int(os.environ.get("RRF_K", "60"))   # reciprocal-rank-fusion constant
HYBRID_BUDGET = int(os.environ.get("HYBRID_BUDGET", "10"))  # evidence items in hybrid ctx

# ---- generation -----------------------------------------------------------
COT_TOK    = int(os.environ.get("COT_TOK", "220"))
ANSWER_TOK = int(os.environ.get("ANSWER_TOK", "320"))
KG_COT     = os.environ.get("KG_COT", "1") == "1"   # per-entity CoT traces in graph ctx
LOAD_4BIT  = os.environ.get("LOAD_4BIT", "1") == "1"

# ---- token ----------------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:                                  # interactive Kaggle fallback
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    raise SystemExit("[fatal] HF_TOKEN missing — pass --secrets HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

# ---- device ---------------------------------------------------------------
ngpu = torch.cuda.device_count()
if ngpu:
    total = sum(torch.cuda.get_device_properties(i).total_memory
                for i in range(ngpu)) / 1e9
    major, _ = torch.cuda.get_device_capability(0)
    DTYPE = torch.bfloat16 if major >= 8 else torch.float16
    print(f"[gpu] {ngpu}x {torch.cuda.get_device_name(0)} (sm_{major}x) | "
          f"{total:.0f}GB | {DTYPE}", flush=True)
    if major < 8:
        print("[gpu] WARNING pre-Ampere card: bf16 unavailable, fp16 overflow is "
              "possible with 4-bit 27B. l4x1 or a10g is safer.", flush=True)
else:
    DTYPE = torch.float32
    print("[gpu] WARNING no GPU — this will be unusably slow", flush=True)

WORK = os.environ.get("WORK_DIR", "/tmp/hybridrag" if ngpu else ".")
os.makedirs(WORK, exist_ok=True)
print(f"[cfg] strategies={STRATEGIES} | LIMIT={LIMIT or 'all'} | "
      f"SC={SC_SAMPLES} | work={WORK}", flush=True)

## 2 · Retriever 1 — the knowledge graph

The graph is pulled as the pickle written by `01_knowledge_graph_construction_and_visualization.ipynb`.

Two retrieval operations are exposed:

- `facts_for(entity)` — the entity's neighbourhood, both in- and out-edges,
  ranked by extraction confidence then by how many papers support the edge.
- `bridges(q_ents, opt_ents)` — edges that *directly connect* a question entity
  to an option entity. These are the discriminative ones: a bridge fact tells the
  model which option is linked to the stem, which a neighbourhood dump does not.
  Bridge coverage was the metric that separated the domain KG (26%) from
  Hetionet (0%), so it is tracked per question here as well.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import networkx as nx

api = HfApi(token=HF_TOKEN)

def _fmt_facts(rows, k):
    """Rank by (confidence, n_papers), dedupe, cap at k."""
    rows.sort(key=lambda x: (-x[0], -x[1]))
    seen, facts = set(), []
    for c, n, src, rel, dst in rows:
        line = f"{src} —{rel}→ {dst}"
        if line in seen: continue
        seen.add(line)
        facts.append(f"{line}  [confidence {c}/10, {n} paper(s)]")
        if len(facts) >= k: break
    return facts

class GraphRetriever:
    name = "networkx (pickle)"
    def __init__(self):
        gp = hf_hub_download(repo_id=KG_REPO, repo_type="dataset",
                             filename="alzheimers_kg.pkl", token=HF_TOKEN)
        blob = pickle.load(open(gp, "rb"))
        self.G = blob["graph"]
        self.disease = blob.get("disease", "the disease")
    def stats(self):      return self.G.number_of_nodes(), self.G.number_of_edges()
    def node_names(self): return list(self.G.nodes())
    def label(self, n):   return self.G.nodes[n].get("label", n)
    def degree(self, n):  return self.G.degree(n)

    def facts_for(self, e, k, mc):
        out = []
        for nbr in self.G.successors(e):
            for _, d in (self.G.get_edge_data(e, nbr) or {}).items():
                if d.get("confidence", 0) >= mc:
                    out.append((d["confidence"], d.get("n_evidence", 1),
                                self.label(e), d["relation"], self.label(nbr)))
        for nbr in self.G.predecessors(e):
            for _, d in (self.G.get_edge_data(nbr, e) or {}).items():
                if d.get("confidence", 0) >= mc:
                    out.append((d["confidence"], d.get("n_evidence", 1),
                                self.label(nbr), d["relation"], self.label(e)))
        return _fmt_facts(out, k)

    def bridges(self, q_ents, opt_ents, mc):
        out = []
        for a in q_ents:
            for b in opt_ents:
                if a == b: continue
                for u, v in ((a, b), (b, a)):
                    for _, d in (self.G.get_edge_data(u, v) or {}).items():
                        if d.get("confidence", 0) >= mc:
                            out.append(f"{self.label(u)} —{d['relation']}→ "
                                       f"{self.label(v)}  [confidence {d['confidence']}/10]")
        return list(dict.fromkeys(out))

KG = GraphRetriever()
DISEASE = KG.disease
_n, _e = KG.stats()
NODE_NAMES = KG.node_names()
print(f"[kg] {DISEASE}: {_n} nodes, {_e} edges", flush=True)

## 3 · Entity linking

The alias table is the same one the graph builder used, so a question saying
"APOE4", "APOE-ε4 allele" or "apolipoprotein E4" resolves to the single merged
node rather than three fragments. Candidates are ranked by node degree —
high-degree entities carry more retrievable evidence, so when a question mentions
several the densest ones are worth the entity budget.

In [ ]:
ALIASES = {
    "alzheimer's disease": ["alzheimer disease","alzheimers disease","alzheimer's",
                            "alzheimers","alzheimer","ad","alzheimer's dementia"],
    "apoe4": ["apoe-4","apoe e4","apoe epsilon 4","apoe epsilon 4 allele",
              "apoe-epsilon 4 allele","apoe ε4","apolipoprotein e4","apoe4 allele"],
    "amyloid beta": ["amyloid-beta","abeta","aβ","amyloid β","amyloid-β",
                     "beta-amyloid","β-amyloid","amyloid beta peptide"],
    "amyloid plaques": ["amyloid plaque","senile plaques","senile plaque"],
    "neurofibrillary tangles": ["neurofibrillary tangle","nfts","nft"],
    "tau protein": ["tau","microtubule-associated protein tau"],
    "dementia": ["dementias","cognitive impairment"],
}
_ALIAS = {}
for _c, _vs in ALIASES.items():
    _ALIAS[_c] = _c
    for _v in _vs:
        _ALIAS[_v] = _c

def norm(t):
    s = re.sub(r"\s+", " ", t.strip().lower()).strip(" .,;:()[]")
    return re.sub(r"^(the|a|an)\s+", "", s)

def lbl(n): return KG.label(n)

def extract_terms(text, max_ents=MAX_ENTITIES):
    tl = norm(text); node_set = set(NODE_NAMES)
    hits = [n for n in NODE_NAMES if len(n) > 4 and n in tl]
    for alias, canon in _ALIAS.items():
        if len(alias) > 3 and alias in tl and canon in node_set:
            hits.append(canon)
    if not hits:                       # fuzzy fallback for unseen spellings
        for w in re.findall(r"[a-z]{5,}", tl)[:10]:
            hits += get_close_matches(w, NODE_NAMES, n=1, cutoff=0.9)
    return sorted(dict.fromkeys(hits), key=lambda n: -KG.degree(n))[:max_ents]

print("[link] alias table:", len(_ALIAS), "surface forms", flush=True)

## 4 · Retriever 2 — the vector index

Loads the FAISS index and the aligned chunk file produced by
`03_vectorrag_pipeline.ipynb`. The index is inner-product over L2-normalised
PubMedBERT embeddings, so the returned score is cosine similarity and is directly
comparable across queries.

The query is the stem concatenated with the options. Retrieving on the stem alone
returns chunks about the topic in general; including the option text lets the
retriever surface passages that discriminate *between* the choices — the vector
analogue of a bridge fact.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

class VectorRetriever:
    name = "faiss (PubMedBERT, cosine)"
    def __init__(self):
        ip = hf_hub_download(repo_id=KG_REPO, repo_type="dataset",
                             filename="ad_faiss.index", token=HF_TOKEN)
        cp = hf_hub_download(repo_id=KG_REPO, repo_type="dataset",
                             filename="ad_chunks.jsonl", token=HF_TOKEN)
        try:
            mp = hf_hub_download(repo_id=KG_REPO, repo_type="dataset",
                                 filename="ad_vector_meta.json", token=HF_TOKEN)
            self.meta = json.load(open(mp))
        except Exception:
            self.meta = {}
        self.index  = faiss.read_index(ip)
        self.chunks = [json.loads(l) for l in open(cp) if l.strip()]
        enc_name = self.meta.get("encoder", ENCODER)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.enc = SentenceTransformer(enc_name, device=dev)
        self.encoder_name = enc_name
        assert self.index.ntotal == len(self.chunks), (
            f"index/chunks misaligned: {self.index.ntotal} vs {len(self.chunks)}")

    def search(self, query, k):
        qv = self.enc.encode([query], convert_to_numpy=True,
                             normalize_embeddings=True)
        sims, idx = self.index.search(qv, k)
        out = []
        for s, i in zip(sims[0], idx[0]):
            if i < 0: continue
            c = self.chunks[i]
            out.append({"score": float(s), "pmid": c.get("pmid", "?"),
                        "text": c["text"], "row": int(i)})
        return out

VEC = VectorRetriever()
print(f"[vec] {VEC.index.ntotal} chunks | encoder {VEC.encoder_name} | "
      f"{VEC.meta.get('n_abstracts','?')} abstracts", flush=True)

# spot-check that retrieval is behaving before spending GPU hours on it
for _q in ["role of APOE4 in Alzheimer's disease", "how tau tangles form"]:
    _h = VEC.search(_q, 1)[0]
    print(f"  '{_q}' -> ({_h['score']:.3f}) {_h['text'][:90]}...", flush=True)

## 5 · Generator

`resolve_generator` prefers a pre-quantised repo: it downloads ~16.5 GB instead
of pulling ~54 GB of fp16 weights and quantising locally, which on a metered Job
is the difference between minutes and most of an hour.

`device_map="auto"` shards if more than one card is present. On a single L4 the
whole 4-bit model fits, which is preferable — sharded bitsandbytes across two
pre-Ampere T4s is where the device-side-assert failures came from.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import model_info

def resolve_generator(requested, want_4bit):
    if not want_4bit:
        return requested, False
    for c in ["unsloth/medgemma-27b-text-it-unsloth-bnb-4bit",
              "unsloth/medgemma-27b-text-it-bnb-4bit",
              "unsloth/medgemma-27b-it-unsloth-bnb-4bit"]:
        try:
            model_info(c, token=HF_TOKEN)
            print(f"[model] using pre-quantised {c}", flush=True)
            return c, True
        except Exception as e:
            print(f"[model]   {c} unavailable ({type(e).__name__})", flush=True)
    print(f"[model] no pre-quantised repo — quantising {requested} on the fly", flush=True)
    return requested, False

MODEL_ID, PREQUANT = resolve_generator(GENERATOR, LOAD_4BIT)
print(f"[model] loading {MODEL_ID} ...", flush=True)

_kw = dict(device_map="auto", token=HF_TOKEN)
if PREQUANT:
    _kw["dtype"] = DTYPE
elif LOAD_4BIT:
    from transformers import BitsAndBytesConfig
    _kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
else:
    _kw["dtype"] = DTYPE

tok   = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_kw)
model.eval()

if hasattr(model, "hf_device_map"):
    print("[model] placement:", dict(Counter(str(v) for v in model.hf_device_map.values())),
          flush=True)
for i in range(torch.cuda.device_count()):
    print(f"[model]   cuda:{i} {torch.cuda.memory_allocated(i)/1e9:.1f}GB", flush=True)
print("[model] ready", flush=True)

## 6 · Generation helpers

`parse_letter` is deliberately layered: it looks for an explicit `Final answer:`
line first, then a bare `Answer:`, then falls back to the last standalone letter
in the trace. Reasoning models often restate options mid-trace, so taking the
*last* match rather than the first is what keeps the parse aligned with the
model's conclusion instead of an intermediate consideration.

`letter_confidence` reads the next-token distribution over the option letters in
a single forward pass. The normalised top-1 minus top-2 gap is the margin. It is
the diagnostic that explains a null result: if the margin is already near 1.0 the
model is certain before seeing any evidence, and retrieval has no headroom to
change the answer regardless of how good the evidence is.

In [ ]:
def gen(prompt, max_new, temperature=None):
    msgs = [{"role": "user", "content": prompt}]
    inp = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    ilen = inp["input_ids"].shape[1]
    kw = dict(max_new_tokens=max_new)
    if temperature and temperature > 0:
        kw.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        kw.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inp, **kw)
    seq = out[0]
    # generate() may return completion-only or prompt+completion depending on
    # config — slicing unconditionally returns "" in the former case
    new = seq[ilen:] if seq.shape[0] > ilen else seq
    return tok.decode(new, skip_special_tokens=True).strip()

def format_options(o):
    return "\n".join(f"{k}. {v}" for k, v in o.items())

def parse_letter(text, letters):
    up = text.upper()
    for pat in (rf"FINAL ANSWER\s*[:\-]?\s*\(?([{letters}])\b",
                rf"\bANSWER\s*[:\-]?\s*\(?([{letters}])\b"):
        m = re.search(pat, up)
        if m: return m.group(1)
    hits = re.findall(rf"\b([{letters}])\b", up)
    if hits: return hits[-1]
    hits = re.findall(rf"([{letters}])", up)
    return hits[-1] if hits else letters[0]

def letter_confidence(question, options):
    letters = list(options.keys())
    prompt = ("You are a biomedical expert. Answer with ONLY the single letter "
              f"({'/'.join(letters)}).\n\nQuestion: {question}\n"
              f"Options:\n{format_options(options)}\nAnswer:")
    msgs = [{"role": "user", "content": prompt}]
    inp = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        logits = model(**inp).logits[0, -1, :]
    probs = torch.softmax(logits.float(), dim=-1)
    scored = []
    for L in letters:
        ids = {tok.encode(v, add_special_tokens=False)[0]
               for v in (L, f" {L}") if tok.encode(v, add_special_tokens=False)}
        scored.append((max((probs[i].item() for i in ids), default=0.0), L))
    scored.sort(reverse=True)
    if len(scored) < 2:
        return scored[0][1], 1.0
    tot = sum(p for p, _ in scored) or 1.0
    return scored[0][1], (scored[0][0] - scored[1][0]) / tot

## 7 · The three retrieval contexts

### GraphRAG
Bridge edges first (they discriminate between options), then per-entity
neighbourhoods. With `KG_COT=1` each entity's fact block is passed back through
the model for a short reasoning trace before the answer prompt — the per-entity
CoT step.

### VectorRAG
Top-*k* chunks for the stem+options query, each labelled with its PMID and
cosine score so the model can weigh them.

### HybridRAG — reciprocal rank fusion
The two retrievers return incomparable scores: a KG edge carries a 1–10
extraction confidence, a chunk carries a cosine similarity. Normalising one onto
the other's scale would be arbitrary. RRF sidesteps this by discarding magnitudes
and fusing on *rank* alone:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

with $k = 60$. An item ranked highly by either retriever scores well; an item
ranked highly by both wins. Because the fused list is truncated to
`HYBRID_BUDGET`, hybrid sees roughly the same *quantity* of evidence as the
single-retriever arms — so any gain reflects better evidence selection, not
simply a longer context.

In [ ]:
def cot_trace(question, entity, facts):
    if not facts: return ""
    return gen(f"Reason about this medical question using verified knowledge-graph facts.\n\n"
               f"Question: {question}\n\nFacts about '{lbl(entity)}':\n"
               + "\n".join(f"- {f}" for f in facts) +
               "\n\nIn 2-3 sentences, reason step by step about what these imply for the "
               "question. Weigh the confidence scores.", COT_TOK)

# ---------------------------------------------------------------- GraphRAG
def graph_evidence(question, options):
    """Returns (ranked evidence strings, detail dict). Rank order = retrieval order."""
    q_ents   = extract_terms(question)
    opt_ents = extract_terms(" ".join(options.values())) if options else []
    bridges  = KG.bridges(q_ents, opt_ents, MIN_CONF)
    ranked, per_ent = list(bridges), {}
    for e in q_ents:
        f = KG.facts_for(e, FACTS_PER_ENT, MIN_CONF)
        per_ent[e] = f
        ranked += f
    ranked = list(dict.fromkeys(ranked))
    return ranked, {"entities": [lbl(e) for e in q_ents],
                    "per_ent": per_ent, "bridges": bridges, "q_ents": q_ents}

def graph_context(question, options, with_cot=KG_COT):
    ranked, d = graph_evidence(question, options)
    blocks = []
    if d["bridges"]:
        blocks.append("Direct links between question and options:\n" +
                      "\n".join(f"- {b}" for b in d["bridges"]))
    for e in d["q_ents"]:
        f = d["per_ent"][e]
        if f:
            blocks.append(f"Facts about {lbl(e)}:\n" + "\n".join(f"- {x}" for x in f))
    traces = []
    if with_cot:
        for e in d["q_ents"]:
            t = cot_trace(question, e, d["per_ent"][e])
            if t: traces.append(f"Reasoning about {lbl(e)}:\n{t}")
    det = {"entities": d["entities"], "facts": ranked, "bridges": d["bridges"],
           "traces": traces, "chunks": [], "fused": []}
    return "\n\n".join(blocks + traces), det

# ---------------------------------------------------------------- VectorRAG
def vector_evidence(question, options, k=VEC_TOPK):
    q = question + ("\n" + " ".join(options.values()) if options else "")
    return VEC.search(q, k)

def vector_context(question, options):
    hits = vector_evidence(question, options)
    block = "Relevant passages from the Alzheimer's literature:\n" + "\n\n".join(
        f"[PMID {h['pmid']}, similarity {h['score']:.3f}]\n{h['text']}" for h in hits)
    det = {"entities": [], "facts": [], "bridges": [], "traces": [],
           "chunks": [{"pmid": h["pmid"], "score": round(h["score"], 4)} for h in hits],
           "fused": []}
    return (block if hits else ""), det

# ---------------------------------------------------------------- HybridRAG
def rrf_fuse(ranked_lists, k=RRF_K, budget=HYBRID_BUDGET):
    """Reciprocal rank fusion over N ranked lists of (key, payload)."""
    score, payload, origin = defaultdict(float), {}, defaultdict(set)
    for src, lst in ranked_lists:
        for rank, (key, pl) in enumerate(lst, start=1):
            score[key] += 1.0 / (k + rank)
            payload.setdefault(key, pl)
            origin[key].add(src)
    order = sorted(score, key=lambda x: -score[x])[:budget]
    return [{"key": key, "score": round(score[key], 5),
             "sources": sorted(origin[key]), "payload": payload[key]} for key in order]

def hybrid_context(question, options):
    g_ranked, gd = graph_evidence(question, options)
    v_hits       = vector_evidence(question, options, k=max(VEC_TOPK, HYBRID_BUDGET))

    g_list = [(f"kg::{f}",  {"kind": "kg",  "text": f}) for f in g_ranked]
    v_list = [(f"vec::{h['row']}", {"kind": "vec", "text": h["text"],
                                    "pmid": h["pmid"], "score": h["score"]})
              for h in v_hits]
    fused = rrf_fuse([("graph", g_list), ("vector", v_list)])

    kg_lines, vec_lines = [], []
    for item in fused:
        p = item["payload"]
        if p["kind"] == "kg":
            kg_lines.append(f"- {p['text']}")
        else:
            vec_lines.append(f"[PMID {p['pmid']}, similarity {p['score']:.3f}]\n{p['text']}")

    blocks = []
    if kg_lines:
        blocks.append("Verified relations from the knowledge graph:\n" + "\n".join(kg_lines))
    if vec_lines:
        blocks.append("Supporting passages from the source abstracts:\n" +
                      "\n\n".join(vec_lines))

    traces = []
    if KG_COT:
        for e in gd["q_ents"]:
            t = cot_trace(question, e, gd["per_ent"][e])
            if t: traces.append(f"Reasoning about {lbl(e)}:\n{t}")

    det = {"entities": gd["entities"], "facts": g_ranked, "bridges": gd["bridges"],
           "traces": traces,
           "chunks": [{"pmid": h["pmid"], "score": round(h["score"], 4)} for h in v_hits],
           "fused": [{"kind": i["payload"]["kind"], "sources": i["sources"],
                      "rrf": i["score"]} for i in fused]}
    return "\n\n".join(blocks + traces), det

CONTEXT_FN = {
    "closed_book": lambda q, o: ("", {"entities": [], "facts": [], "bridges": [],
                                      "traces": [], "chunks": [], "fused": []}),
    "graphrag":    graph_context,
    "vectorrag":   vector_context,
    "hybridrag":   hybrid_context,
}
print("[ctx] strategies wired:", list(CONTEXT_FN), flush=True)

## 8 · Answer

One prompt template across all four arms. This matters: if graph and vector used
differently-worded instructions, part of any accuracy gap would be prompt
engineering rather than retrieval quality. Only the `Context:` block differs.

The instruction explicitly permits falling back on parametric knowledge when the
context is unhelpful. Without that licence the model tends to force an answer out
of irrelevant evidence, which turns a retrieval miss into a wrong answer instead
of a graceful no-op.

In [ ]:
def build_prompt(question, options, ctx):
    letters = "".join(options.keys())
    return ("You are a biomedical expert specialising in neurodegenerative disease. "
            "Use the context where relevant; if it does not address the question, "
            "rely on your own medical knowledge.\n\n"
            f"Context:\n{ctx if ctx.strip() else '(none)'}\n\n"
            f"Question: {question}\nOptions:\n{format_options(options)}\n\n"
            "Reason briefly, then end with a line exactly like:\n"
            f"Final answer: <one letter {'/'.join(letters)}>")

def answer(question, options, strategy):
    letters = "".join(options.keys())
    t0 = time.time()
    ctx, det = CONTEXT_FN[strategy](question, options)
    det["votes"] = None; det["vote_margin"] = None
    det["ctx_chars"] = len(ctx)
    prompt = build_prompt(question, options, ctx)

    if SC_SAMPLES > 1:
        votes = [parse_letter(gen(prompt, ANSWER_TOK, temperature=SC_TEMP), letters)
                 for _ in range(SC_SAMPLES)]
        cnt = Counter(votes).most_common()
        det["votes"] = dict(Counter(votes))
        det["vote_margin"] = (cnt[0][1] - (cnt[1][1] if len(cnt) > 1 else 0)) / SC_SAMPLES
        pred = cnt[0][0]
    else:
        pred = parse_letter(gen(prompt, ANSWER_TOK), letters)
    det["seconds"] = round(time.time() - t0, 2)
    return pred, det

## 9 · Questions

Prefers `ad_questions.json` written by notebook 01 so the evaluation set is
byte-identical to the one the graph's coverage was measured against. Falls back to
filtering MIRAGE directly if that file is absent.

In [ ]:
try:
    qp = hf_hub_download(repo_id=KG_REPO, repo_type="dataset",
                         filename="ad_questions.json", token=HF_TOKEN)
    QUESTIONS = json.load(open(qp))
    print("[data] loaded ad_questions.json from the KG repo", flush=True)
except Exception:
    print("[data] ad_questions.json not found — filtering MIRAGE directly", flush=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Teddy-XiongGZ/MIRAGE/main/benchmark.json",
        f"{WORK}/mirage.json")
    raw = json.load(open(f"{WORK}/mirage.json"))
    pat = re.compile(os.environ.get(
        "AD_PATTERN", r"alzheimer|dementia|amyloid|tau protein|neurodegenerat"), re.I)
    QUESTIONS = [{"benchmark": b, "id": qid, "question": it["question"],
                  "options": it.get("options", {}), "answer": it.get("answer", "")}
                 for b, items in raw.items() for qid, it in items.items()
                 if pat.search(it["question"] + " " + " ".join(it.get("options", {}).values()))]

QUESTIONS = [q for q in QUESTIONS if q.get("options") and q.get("answer")]
if LIMIT: QUESTIONS = QUESTIONS[:LIMIT]
N = len(QUESTIONS)
print(f"[data] {N} scorable questions | "
      f"{dict(Counter(q['benchmark'] for q in QUESTIONS))}", flush=True)

## 10 · Retrieval coverage — before any generation

This is cheap (no GPU generation) and worth running first: it says whether the
retrievers can reach the questions at all. If graph bridge coverage is near zero
the GraphRAG arm cannot beat the baseline no matter how good the generator is,
and knowing that now saves the hours of inference it would take to find out.

In [ ]:
cov = {"n": N, "kg_entity_linked": 0, "kg_any_fact": 0, "kg_bridge": 0,
       "vec_hits": 0, "vec_mean_top1": []}
for q in QUESTIONS:
    ents = extract_terms(q["question"])
    if ents: cov["kg_entity_linked"] += 1
    g_ranked, gd = graph_evidence(q["question"], q["options"])
    if g_ranked:      cov["kg_any_fact"] += 1
    if gd["bridges"]: cov["kg_bridge"] += 1
    hits = vector_evidence(q["question"], q["options"], k=1)
    if hits:
        cov["vec_hits"] += 1
        cov["vec_mean_top1"].append(hits[0]["score"])

cov["vec_mean_top1"] = round(float(np.mean(cov["vec_mean_top1"])), 4) if cov["vec_mean_top1"] else 0.0
print("\n===== RETRIEVAL COVERAGE =====", flush=True)
print(f"KG entity-linked   : {cov['kg_entity_linked']}/{N} ({cov['kg_entity_linked']/N:.0%})", flush=True)
print(f"KG >=1 fact        : {cov['kg_any_fact']}/{N} ({cov['kg_any_fact']/N:.0%})", flush=True)
print(f"KG bridge fact     : {cov['kg_bridge']}/{N} ({cov['kg_bridge']/N:.0%})", flush=True)
print(f"Vector top-1 hit   : {cov['vec_hits']}/{N}  mean cosine {cov['vec_mean_top1']}", flush=True)
if cov["kg_bridge"] == 0:
    print("\n*** No bridge facts. GraphRAG has nothing discriminative to offer —\n"
          "*** expect it to match the baseline. Densify the graph first.", flush=True)

## 11 · Run

Closed-book letter margins are measured once, up front, and reused across all
strategies — the margin is a property of the question and the model, not of the
retrieval arm, so recomputing it per arm would just burn forward passes.

In [ ]:
CONF = {}
print("[run] measuring closed-book letter margins ...", flush=True)
_t0 = time.time()
for q in QUESTIONS:
    _, m = letter_confidence(q["question"], q["options"])
    CONF[q["id"]] = round(m, 4)
_mean_margin = float(np.mean(list(CONF.values()))) if CONF else 0.0
print(f"[run] mean margin {_mean_margin:.3f}  ({time.time()-_t0:.0f}s)", flush=True)
print("[run]   1.0 = certain before retrieval; high values cap how much "
      "retrieval can possibly change", flush=True)

results, records = {}, []
for strat in STRATEGIES:
    t0, correct = time.time(), 0
    for i, q in enumerate(QUESTIONS, 1):
        pred, det = answer(q["question"], q["options"], strat)
        gold = q["answer"].strip().upper()
        ok = (pred == gold); correct += ok
        records.append({
            "strategy": strat, "benchmark": q["benchmark"], "id": q["id"],
            "question": q["question"][:160], "pred": pred, "gold": gold,
            "correct": bool(ok), "margin": CONF[q["id"]],
            "n_entities": len(det["entities"]), "n_facts": len(det["facts"]),
            "n_bridges": len(det["bridges"]), "n_traces": len(det["traces"]),
            "n_chunks": len(det["chunks"]),
            "n_fused_kg":  sum(1 for f in det["fused"] if f["kind"] == "kg"),
            "n_fused_vec": sum(1 for f in det["fused"] if f["kind"] == "vec"),
            "ctx_chars": det["ctx_chars"], "seconds": det["seconds"],
            "votes": det["votes"], "vote_margin": det["vote_margin"]})
        if i % 5 == 0 or i == N:
            rate = (time.time() - t0) / i
            print(f"  [{strat}] {i}/{N} | acc {correct/i:.3f} | {rate:.0f}s/q | "
                  f"ETA {rate*(N-i)/60:.0f} min", flush=True)
    results[strat] = {"accuracy": round(correct / max(N, 1), 4), "n": N,
                      "correct": correct,
                      "minutes": round((time.time() - t0) / 60, 1)}
    print(f"[run] {strat}: {results[strat]['accuracy']:.4f} "
          f"({correct}/{N}) in {results[strat]['minutes']} min", flush=True)

## 12 · Statistics

**Wilson intervals**, not normal-approximation ones — at n≈54 the normal
approximation is unreliable near the tails and can produce bounds outside [0,1].

**McNemar** against `closed_book`, because every arm answers the *same*
questions. Treating two accuracies on identical items as independent samples
overstates the evidence; McNemar looks only at the disagreements — how many the
retriever fixed (b01) versus broke (b10).

**Fixed/broke** is reported alongside the net delta because they can cancel. An
arm that fixes 6 and breaks 6 nets to zero but is not behaving like the baseline,
and that distinction is invisible in the accuracy column alone.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def mcnemar(a, b):
    ca = {r["id"]: r["correct"] for r in records if r["strategy"] == a}
    cb = {r["id"]: r["correct"] for r in records if r["strategy"] == b}
    ids = set(ca) & set(cb)
    b01 = sum(1 for i in ids if not ca[i] and cb[i])   # b fixed what a missed
    b10 = sum(1 for i in ids if ca[i] and not cb[i])   # b broke what a had
    n = b01 + b10
    if n == 0: return b01, b10, 0.0, 1.0
    chi = (abs(b01 - b10) - 1) ** 2 / n
    p = math.erfc(math.sqrt(chi / 2)) if chi > 0 else 1.0
    return b01, b10, chi, p

BASE = STRATEGIES[0]
stats = {}
for s in STRATEGIES:
    lo, hi = wilson(results[s]["correct"], N)
    stats[s] = {"acc": results[s]["accuracy"], "ci": [round(lo, 4), round(hi, 4)]}
    if s != BASE:
        b01, b10, chi, p = mcnemar(BASE, s)
        stats[s]["vs_base"] = {"fixed": b01, "broke": b10,
                               "chi2": round(chi, 3), "p": round(p, 4),
                               "delta": round(results[s]["accuracy"] -
                                              results[BASE]["accuracy"], 4)}

# head-to-head between the retrieval arms (does hybrid beat its parts?)
pairs = {}
for a in STRATEGIES:
    for b in STRATEGIES:
        if a >= b or a == BASE: continue
        b01, b10, chi, p = mcnemar(a, b)
        pairs[f"{a}_vs_{b}"] = {"fixed": b01, "broke": b10,
                                "chi2": round(chi, 3), "p": round(p, 4)}

by_q   = defaultdict(dict)
for r in records: by_q[r["id"]][r["strategy"]] = r
never  = [i for i, d in by_q.items() if not any(x["correct"] for x in d.values())]
always = [i for i, d in by_q.items() if all(x["correct"] for x in d.values())]
# questions only one arm gets right — the interesting ones for error analysis
unique = {s: [i for i, d in by_q.items()
              if d.get(s, {}).get("correct") and
              sum(1 for x in d.values() if x["correct"]) == 1]
          for s in STRATEGIES}

W = 74
print("\n" + "=" * W, flush=True)
print("HYBRIDRAG EVALUATION — GraphRAG vs VectorRAG vs Hybrid", flush=True)
print("=" * W, flush=True)
print(f"generator      : {MODEL_ID}{'  [pre-quantised 4-bit]' if PREQUANT else ''}", flush=True)
print(f"knowledge graph: {_n} nodes / {_e} edges", flush=True)
print(f"vector index   : {VEC.index.ntotal} chunks / {VEC.encoder_name}", flush=True)
print(f"questions      : {N} | self-consistency {SC_SAMPLES} | mean margin {_mean_margin:.3f}", flush=True)
print("-" * W, flush=True)
print(f"{'strategy':<14}{'acc':>8}{'95% CI':>18}{'fixed':>8}{'broke':>8}{'p':>9}", flush=True)
for s in STRATEGIES:
    st = stats[s]
    ci = f"[{st['ci'][0]:.3f},{st['ci'][1]:.3f}]"
    if "vs_base" in st:
        v = st["vs_base"]
        print(f"{s:<14}{st['acc']:>8.4f}{ci:>18}{v['fixed']:>8}{v['broke']:>8}{v['p']:>9.4f}", flush=True)
    else:
        print(f"{s:<14}{st['acc']:>8.4f}{ci:>18}{'—':>8}{'—':>8}{'(base)':>9}", flush=True)
print("-" * W, flush=True)
for k, v in pairs.items():
    print(f"{k:<26} fixed {v['fixed']:>3}  broke {v['broke']:>3}  p={v['p']:.4f}", flush=True)
print("-" * W, flush=True)
print(f"never solved by any arm : {len(never)}/{N}", flush=True)
print(f"solved by every arm     : {len(always)}/{N}", flush=True)
for s in STRATEGIES:
    print(f"  uniquely solved by {s:<12}: {len(unique[s])}", flush=True)

# evidence-volume audit — confirms hybrid isn't just winning on context length
print("\nevidence supplied per strategy (mean):", flush=True)
for s in STRATEGIES:
    rs = [r for r in records if r["strategy"] == s]
    if not rs: continue
    print(f"  {s:<12} facts {np.mean([r['n_facts'] for r in rs]):5.1f} | "
          f"bridges {np.mean([r['n_bridges'] for r in rs]):4.1f} | "
          f"chunks {np.mean([r['n_chunks'] for r in rs]):4.1f} | "
          f"ctx {np.mean([r['ctx_chars'] for r in rs]):6.0f} chars | "
          f"{np.mean([r['seconds'] for r in rs]):5.1f} s/q", flush=True)

best = max(STRATEGIES, key=lambda s: stats[s]["acc"])
print(f"\nbest: {best} = {stats[best]['acc']:.4f}", flush=True)
if best != BASE and stats[best].get("vs_base", {}).get("p", 1) > 0.05:
    print("NOTE: the best arm's CI overlaps the baseline and McNemar p > 0.05. "
          "At this n the result is suggestive, not significant.", flush=True)

## 13 · Figures

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGDIR = os.path.join(WORK, "Results")
os.makedirs(FIGDIR, exist_ok=True)
COLORS = {"closed_book": "#6c757d", "graphrag": "#1f4e79",
          "vectorrag": "#c0392b", "hybridrag": "#1e8449"}

# --- accuracy with Wilson bars ---------------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 4.2))
xs = list(STRATEGIES)
ys = [stats[s]["acc"] for s in xs]
lo = [stats[s]["acc"] - stats[s]["ci"][0] for s in xs]
hi = [stats[s]["ci"][1] - stats[s]["acc"] for s in xs]
ax.bar(xs, ys, color=[COLORS.get(s, "#888") for s in xs], width=.6)
ax.errorbar(xs, ys, yerr=[lo, hi], fmt="none", ecolor="black", capsize=5, lw=1.2)
ax.axhline(stats[BASE]["acc"], ls="--", c="black", lw=1,
           label=f"closed-book = {stats[BASE]['acc']:.3f}")
for i, (x, y) in enumerate(zip(xs, ys)):
    ax.text(i, y + hi[i] + .015, f"{y:.3f}", ha="center", fontsize=9)
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1)
ax.set_title(f"Retrieval strategy vs accuracy  (n={N}, 95% Wilson CI)")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=.25)
fig.tight_layout(); fig.savefig(f"{FIGDIR}/accuracy_by_strategy.png", dpi=160)
plt.close(fig)

# --- fixed / broke ----------------------------------------------------------
arms = [s for s in STRATEGIES if s != BASE]
if arms:
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    w = .35; idx = np.arange(len(arms))
    fx = [stats[s]["vs_base"]["fixed"] for s in arms]
    bk = [stats[s]["vs_base"]["broke"] for s in arms]
    ax.bar(idx - w/2, fx, w, label="fixed vs closed-book", color="#1e8449")
    ax.bar(idx + w/2, bk, w, label="broke vs closed-book", color="#c0392b")
    ax.set_xticks(idx); ax.set_xticklabels(arms)
    ax.set_ylabel("questions"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=.25)
    ax.set_title("Where retrieval changed the answer")
    fig.tight_layout(); fig.savefig(f"{FIGDIR}/fixed_vs_broke.png", dpi=160)
    plt.close(fig)

# --- accuracy vs closed-book certainty -------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 4.0))
edges = [0, .5, .8, .95, 1.01]
labels = ["<0.5", "0.5-0.8", "0.8-0.95", ">0.95"]
for s in STRATEGIES:
    rs = [r for r in records if r["strategy"] == s]
    ys2 = []
    for a, b in zip(edges[:-1], edges[1:]):
        sub = [r for r in rs if a <= r["margin"] < b]
        ys2.append(np.mean([r["correct"] for r in sub]) if sub else np.nan)
    ax.plot(labels, ys2, marker="o", label=s, color=COLORS.get(s, "#888"))
ax.set_xlabel("closed-book letter margin (model certainty before retrieval)")
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.02)
ax.set_title("Retrieval helps most where the model was least certain")
ax.legend(fontsize=8); ax.grid(alpha=.25)
fig.tight_layout(); fig.savefig(f"{FIGDIR}/accuracy_by_margin.png", dpi=160)
plt.close(fig)

# --- hybrid fusion composition ---------------------------------------------
if "hybridrag" in STRATEGIES:
    hs = [r for r in records if r["strategy"] == "hybridrag"]
    if hs:
        fig, ax = plt.subplots(figsize=(7.5, 3.6))
        kg  = np.mean([r["n_fused_kg"]  for r in hs])
        vec = np.mean([r["n_fused_vec"] for r in hs])
        ax.barh(["fused context"], [kg],  color="#1f4e79", label=f"KG triples ({kg:.1f})")
        ax.barh(["fused context"], [vec], left=[kg], color="#c0392b",
                label=f"vector chunks ({vec:.1f})")
        ax.set_xlabel(f"mean evidence items (budget {HYBRID_BUDGET})")
        ax.set_title("What RRF actually selected into the hybrid context")
        ax.legend(fontsize=8)
        fig.tight_layout(); fig.savefig(f"{FIGDIR}/hybrid_fusion_mix.png", dpi=160)
        plt.close(fig)

print("[fig] wrote figures to", FIGDIR, flush=True)
print("     ", os.listdir(FIGDIR), flush=True)

## 14 · Report → PDF

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

def textpage(pdf, title, lines, size=8):
    fig = plt.figure(figsize=(8.5, 11))
    fig.text(0.06, 0.95, title, size=14, weight="bold")
    y = 0.91
    for ln in lines[:78]:
        fig.text(0.06, y, ln[:110], size=size, family="monospace"); y -= 0.0115
    pdf.savefig(fig); plt.close(fig)

PDF = os.path.join(FIGDIR, "hybridrag_eval_report.pdf")
with PdfPages(PDF) as pdf:
    fig = plt.figure(figsize=(8.5, 11))
    fig.text(0.5, 0.74, "Alzheimer's HybridRAG", size=24, weight="bold", ha="center")
    fig.text(0.5, 0.69, "GraphRAG vs VectorRAG vs Hybrid", size=14, ha="center")
    cover = [f"Generated: {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}",
             f"Generator: {MODEL_ID}",
             f"Knowledge graph: {_n} nodes / {_e} edges",
             f"Vector index: {VEC.index.ntotal} chunks ({VEC.encoder_name})",
             f"Questions: {N} Alzheimer's MIRAGE items",
             f"Strategies: {', '.join(STRATEGIES)}",
             f"Self-consistency: {SC_SAMPLES} @ T={SC_TEMP}" if SC_SAMPLES > 1 else
             "Self-consistency: off (greedy)",
             "",
             f"Best: {best} = {stats[best]['acc']:.4f}",
             f"Baseline: {stats[BASE]['acc']:.4f}"]
    y = 0.58
    for c in cover:
        fig.text(0.5, y, c, size=10, ha="center"); y -= 0.026
    pdf.savefig(fig); plt.close(fig)

    hdr = f"{'strategy':<14}{'acc':>8}{'95% CI':>18}{'fixed':>8}{'broke':>8}{'p':>9}"
    rows = [hdr, "-" * 66]
    for s in STRATEGIES:
        st = stats[s]
        ci = f"[{st['ci'][0]:.3f},{st['ci'][1]:.3f}]"
        if "vs_base" in st:
            v = st["vs_base"]
            rows.append(f"{s:<14}{st['acc']:>8.4f}{ci:>18}{v['fixed']:>8}"
                        f"{v['broke']:>8}{v['p']:>9.4f}")
        else:
            rows.append(f"{s:<14}{st['acc']:>8.4f}{ci:>18}{'-':>8}{'-':>8}{'base':>9}")
    rows += ["", "head-to-head:"] + [
        f"  {k:<26} fixed {v['fixed']:>3}  broke {v['broke']:>3}  p={v['p']:.4f}"
        for k, v in pairs.items()]
    rows += ["", "retrieval coverage:",
             f"  KG entity-linked {cov['kg_entity_linked']}/{N}",
             f"  KG bridge facts  {cov['kg_bridge']}/{N}",
             f"  vector mean top-1 cosine {cov['vec_mean_top1']}"]
    textpage(pdf, "Results", rows)

    for f in ["accuracy_by_strategy.png", "fixed_vs_broke.png",
              "accuracy_by_margin.png", "hybrid_fusion_mix.png"]:
        p = os.path.join(FIGDIR, f)
        if not os.path.exists(p): continue
        fig = plt.figure(figsize=(8.5, 11))
        fig.figimage(plt.imread(p), 60, 400, zorder=1)
        fig.text(0.06, 0.95, f.replace("_", " ").replace(".png", ""),
                 size=13, weight="bold")
        pdf.savefig(fig); plt.close(fig)

    # per-question errors, grouped by how the arms disagreed
    lines = []
    for qid, d in list(by_q.items())[:60]:
        pat = "".join("1" if d.get(s, {}).get("correct") else "0" for s in STRATEGIES)
        if pat.count("1") in (0, len(STRATEGIES)): continue
        r0 = next(iter(d.values()))
        lines.append(f"[{pat}] {r0['benchmark']:<9} {r0['question'][:78]}")
        lines.append(f"        gold {r0['gold']} | " +
                     " ".join(f"{s[:5]}={d.get(s,{}).get('pred','-')}" for s in STRATEGIES))
    textpage(pdf, f"Disagreements  (bit order: {', '.join(STRATEGIES)})", lines, size=7)

print("[pdf]", PDF, flush=True)

## 15 · Artifacts → `Results/` and the Hub

In [ ]:
RES_JSON = os.path.join(FIGDIR, "hybridrag_eval_results.json")
RES_CSV  = os.path.join(FIGDIR, "hybridrag_eval_predictions.csv")

json.dump({"config": {"generator": MODEL_ID, "requested": GENERATOR,
                      "encoder": VEC.encoder_name, "quantised_4bit": LOAD_4BIT,
                      "prequantised": PREQUANT, "strategies": list(STRATEGIES),
                      "sc_samples": SC_SAMPLES, "kg_cot": KG_COT,
                      "vec_topk": VEC_TOPK, "rrf_k": RRF_K,
                      "hybrid_budget": HYBRID_BUDGET, "min_conf": MIN_CONF,
                      "kg_nodes": _n, "kg_edges": _e,
                      "vec_chunks": int(VEC.index.ntotal),
                      "n_questions": N, "platform": "hf-jobs"},
           "coverage": cov, "results": results, "stats": stats,
           "head_to_head": pairs, "mean_margin": round(_mean_margin, 4),
           "never_solved": never, "always_solved": always,
           "uniquely_solved": unique, "records": records},
          open(RES_JSON, "w"), indent=2)

with open(RES_CSV, "w", newline="") as f:
    cols = [k for k in records[0].keys() if k != "votes"]
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader()
    for r in records:
        w.writerow({k: v for k, v in r.items() if k != "votes"})

print("[out] wrote:", flush=True)
for p in sorted(os.listdir(FIGDIR)):
    print("   ", os.path.join(FIGDIR, p), flush=True)

try:
    api.create_repo(RESULTS_REPO, repo_type="dataset", exist_ok=True)
    for p in sorted(os.listdir(FIGDIR)):
        api.upload_file(path_or_fileobj=os.path.join(FIGDIR, p),
                        path_in_repo=f"results/{p}",
                        repo_id=RESULTS_REPO, repo_type="dataset")
    print(f"[out] pushed to https://huggingface.co/datasets/{RESULTS_REPO}", flush=True)
except Exception as e:
    print(f"[out] push failed ({type(e).__name__}) — files are in {FIGDIR}", flush=True)

## 16 · Reading the result

Three outcomes and what each means:

**Hybrid wins, and McNemar p < 0.05.** The retrievers are complementary — the
`hybrid_fusion_mix` figure will show RRF pulling from both sides, and
`uniquely_solved` will show questions each arm gets alone.

**All arms land on the baseline.** Check `mean_margin` first. If it is above ~0.95
the model is already certain before retrieval and there is no headroom to
measure; that is a property of the question set, not a failure of retrieval. The
fix is harder questions, not better evidence. This is what the earlier
closed-book/kg_direct/kg_cot tie was diagnosing.

**Graph beats vector or vice versa but hybrid beats neither.** Fusion is diluting
the stronger retriever. Cut `HYBRID_BUDGET`, or weight the RRF terms rather than
summing them equally.

Whatever the outcome, the honest framing goes in the README: at n≈54 a difference
needs to be roughly 12 points to clear significance, so a 3–5 point gap is
suggestive at best and the confidence intervals will say so.

---

### Prerequisites

| Artifact | Produced by | Repo path |
|---|---|---|
| `alzheimers_kg.pkl` | `01_knowledge_graph_construction_and_visualization.ipynb` | `KG_REPO` |
| `ad_questions.json` | `01_...ipynb` | `KG_REPO` |
| `ad_faiss.index`, `ad_chunks.jsonl`, `ad_vector_meta.json` | `03_vectorrag_pipeline.ipynb` | `KG_REPO` |

Run 01 and 03 before this notebook.